# First Attempt

In [2]:
import torch
import numpy as np

## Load/ Preprocess Data

In [3]:
import pandas as pd
import numpy as np
import wfdb
import ast

#provided function to load data with information in the agg_df (modified to return a numpy array)
def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data]).astype(np.float32)
    return data

# path to dta and selected sampling rate
path = "physionet.org/files/ptb-xl/1.0.3/"
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x)) #changes class distribution to dict

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1] #only take diagnostic scps

classes = agg_df.index.tolist()

# encodes the labeled as a 44d vector of 1's and zeros (based off of classes list)
def encode_multilabel(class_list):
    return np.array([1 if cls in class_list else 0 for cls in classes])

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] > 50:
                tmp.append(key)
    return list(set(tmp))

# Apply diagnostic superclass and encoding multilabel
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic).apply(encode_multilabel)


In [4]:
Y["diagnostic_superclass"]
Y

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr,diagnostic_superclass
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21833,17180.0,67.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-31 09:14:35,ventrikulÄre extrasystole(n) sinustachykardie ...,...,NaN,", alles,",NaN,NaN,1ES,NaN,7,records100/21000/21833_lr,records500/21000/21833_hr,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
21834,20703.0,300.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-06-05 11:33:39,sinusrhythmus lagetyp normal qrs(t) abnorm ...,...,NaN,NaN,NaN,NaN,NaN,NaN,4,records100/21000/21834_lr,records500/21000/21834_hr,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
21835,19311.0,59.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-08 10:30:27,sinusrhythmus lagetyp normal t abnorm in anter...,...,NaN,", I-AVR,",NaN,NaN,NaN,NaN,2,records100/21000/21835_lr,records500/21000/21835_hr,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [5]:
Y.scp_codes

ecg_id
1                 {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                             {'NORM': 80.0, 'SBRAD': 0.0}
3                               {'NORM': 100.0, 'SR': 0.0}
4                               {'NORM': 100.0, 'SR': 0.0}
5                               {'NORM': 100.0, 'SR': 0.0}
                               ...                        
21833    {'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'ST...
21834             {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21835                           {'ISCAS': 50.0, 'SR': 0.0}
21836                           {'NORM': 100.0, 'SR': 0.0}
21837                           {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 21799, dtype: object

In [6]:
agg_df.index

Index(['NDT', 'NST_', 'DIG', 'LNGQT', 'NORM', 'IMI', 'ASMI', 'LVH', 'LAFB',
       'ISC_', 'IRBBB', '1AVB', 'IVCD', 'ISCAL', 'CRBBB', 'CLBBB', 'ILMI',
       'LAO/LAE', 'AMI', 'ALMI', 'ISCIN', 'INJAS', 'LMI', 'ISCIL', 'LPFB',
       'ISCAS', 'INJAL', 'ISCLA', 'RVH', 'ANEUR', 'RAO/RAE', 'EL', 'WPW',
       'ILBBB', 'IPLMI', 'ISCAN', 'IPMI', 'SEHYP', 'INJIN', 'INJLA', 'PMI',
       '3AVB', 'INJIL', '2AVB'],
      dtype='str')

In [7]:
# Further example code 
"""
Y_s = Y.head(10).copy()

# Load raw signal data
X = load_raw_data(Y_s, sampling_rate, path)

# Split data into train and test
test_fold = 10
# Train
X_train = X[np.where(Y_s.strat_fold != test_fold)]
y_train = Y_s[(Y_s.strat_fold != test_fold)].diagnostic_superclass
# Test
X_test = X[np.where(Y_s.strat_fold == test_fold)]
y_test = Y_s[Y_s.strat_fold == test_fold].diagnostic_superclass"""

'\nY_s = Y.head(10).copy()\n\n# Load raw signal data\nX = load_raw_data(Y_s, sampling_rate, path)\n\n# Split data into train and test\ntest_fold = 10\n# Train\nX_train = X[np.where(Y_s.strat_fold != test_fold)]\ny_train = Y_s[(Y_s.strat_fold != test_fold)].diagnostic_superclass\n# Test\nX_test = X[np.where(Y_s.strat_fold == test_fold)]\ny_test = Y_s[Y_s.strat_fold == test_fold].diagnostic_superclass'

In [8]:
#Y_s.diagnostic_superclass.to_numpy() #example line run of multilabel distribution
#Y['scp_codes']

In [9]:
Y['scp_codes']

ecg_id
1                 {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                             {'NORM': 80.0, 'SBRAD': 0.0}
3                               {'NORM': 100.0, 'SR': 0.0}
4                               {'NORM': 100.0, 'SR': 0.0}
5                               {'NORM': 100.0, 'SR': 0.0}
                               ...                        
21833    {'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'ST...
21834             {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21835                           {'ISCAS': 50.0, 'SR': 0.0}
21836                           {'NORM': 100.0, 'SR': 0.0}
21837                           {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 21799, dtype: object

## Data Prep

In [10]:
from sklearn.model_selection import train_test_split

Y_s = Y.sample(20000)

Y_transformed = np.stack(Y_s.diagnostic_superclass.values).astype(np.float32)

# Load raw signal data
X = load_raw_data(Y_s, sampling_rate, path)

X = np.transpose(X, (0, 2, 1))

X_train, X_test, y_train, y_test = train_test_split(X, Y_transformed, test_size=0.1, random_state=56)

In [11]:
X_train_torch = torch.tensor(X_train) # change dim ordering for PyTorch
y_train_torch = torch.tensor(y_train)
X_test_torch = torch.tensor(X_test) # change dim ordering for PyTorch
y_test_torch = torch.tensor(y_test)

In [12]:
train_dataset = torch.utils.data.TensorDataset(X_train_torch, y_train_torch)
train_dataloader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

test_dataset = torch.utils.data.TensorDataset(X_test_torch, y_test_torch)
test_dataloader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=32)


## CNN

In [13]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


class CNet(nn.Module):
  def __init__(self):
    super(CNet,self).__init__()
    #starting with 96

    self.conv11 = nn.Conv1d(12, 64, 9, padding=4)
    self.conv12 = nn.Conv1d(64, 64, 9, padding=4)
    self.bn1 = nn.BatchNorm1d(64)

    self.pool = nn.MaxPool1d(kernel_size=2, stride = 2)

    self.conv21 = nn.Conv1d(64, 128, 9, padding=4)
    self.conv22 = nn.Conv1d(128, 128, 9, padding=4)
    self.bn2 = nn.BatchNorm1d(128)

    self.conv31 = nn.Conv1d(128, 256, 9, padding=4)
    self.conv32 = nn.Conv1d(256, 256, 9, padding=4)
    self.conv33 = nn.Conv1d(256, 256, 9, padding=4)
    self.bn3 = nn.BatchNorm1d(256)

    self.conv41 = nn.Conv1d(256, 512, 9, padding=4)
    self.conv42 = nn.Conv1d(512, 512, 9, padding=4)
    self.conv43 = nn.Conv1d(512, 512, 9, padding=4)
    self.bn4 = nn.BatchNorm1d(512)

    self.fc1 = nn.Linear(512 * 125, 1000)
    self.fc2 = nn.Linear(1000, 500)
    self.fc3 = nn.Linear(500, 44)

  def forward(self, x):
    #x = self.pool(F.relu(self.conv12(F.relu(self.conv11(x)))))
    #x = self.pool(F.relu(self.conv22(F.relu(self.conv21(x)))))
    #x = self.pool(F.relu(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x)))))))
    #x = F.relu(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x))))))

    x = self.pool(F.relu(self.bn1(self.conv12(F.relu(self.conv11(x))))))
    x = self.pool(F.relu(self.bn2(self.conv22(F.relu(self.conv21(x))))))
    x = self.pool(F.relu(self.bn3(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x))))))))
    x = F.relu(self.bn4(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x)))))))
    x = torch.flatten(x,1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

In [14]:
def m_train_model(model, train_dataloader, test_dataloader, epochs, weight_decay):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.train()

    #loss_fn = torch.nn.CrossEntropyLoss()
    #loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.Tensor([5]).to(device))

    # MASKED LOSS 
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.Tensor([5]).to(device), reduction="none").to(device)

    lr = 1e-3
    opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    #opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    #scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_test = 0
    best_test_train = 0
    best_epoch = 0

    for epoch in range(epochs):
        running_loss = 0
        for batch_index, (X, y) in enumerate(train_dataloader):
            print(f"Epoch: {epoch} Batch {batch_index}")
            X,y = X.to(device), y.to(device)

            opt.zero_grad() #zero out the gradients

            z = model(X)
            loss = loss_fn(z,y) #compute loss
            weights = torch.ones(y.shape[1], device=device, dtype=loss.dtype)
            weights[4] = 0.1
            weighted_loss = (loss * weights.unsqueeze(0))
            #running_loss += loss.item()

            final_loss = weighted_loss.sum() / (weights.sum() * X.size(0))
            running_loss += final_loss.item()
            final_loss.backward()
            #loss.backward() #compute gradients

            opt.step() #apply gradients
            #scheduler.step()



        running_loss = running_loss / len(train_dataloader)
        print(f"Epoch {epoch} train loss: {running_loss}")

def train_model(model, train_dataloader, test_dataloader, epochs, weight_decay):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.train()

    #loss_fn = torch.nn.CrossEntropyLoss()
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.Tensor([5]).to(device))

    lr = 1e-3
    opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    #opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    #scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_test = 0
    best_test_train = 0
    best_epoch = 0

    for epoch in range(epochs):
        running_loss = 0
        for batch_index, (X, y) in enumerate(train_dataloader):
            #print(f"Epoch: {epoch} Batch {batch_index}")
            X,y = X.to(device), y.to(device)

            opt.zero_grad() #zero out the gradients

            z = model(X)
            loss = loss_fn(z,y) #compute loss
            running_loss += loss.item()

            loss.backward() #compute gradients

            opt.step() #apply gradients
            #scheduler.step()



        running_loss = running_loss / len(train_dataloader)
        print(f"Epoch {epoch} train loss: {running_loss}")


In [15]:
X.shape

(20000, 12, 1000)

In [16]:
def jaccard_score(y_pred, y_true):
    intersection = ((y_pred == 1) & (y_true == 1)).sum()
    union = ((y_pred == 1) | (y_true == 1)).sum()

    if union == 0:
        return 0.0
    
    return intersection / union

def precision(y_pred, y_true):
    f_p = ((y_pred == 1) & (y_true == 1)).sum()
    t_p = ((y_pred == 1) & (y_true == 0)).sum()
    if t_p + f_p == 0:
        return 0.0
    return t_p / (t_p + f_p)

def recall(y_pred, y_true):
    t_p = ((y_pred == 1) & (y_true == 1)).sum()
    f_n = ((y_pred == 0) & (y_true == 1)).sum()

    if t_p + f_n == 0:
        return 0.0
    return t_p / (t_p + f_n)

def f1(y_pred, y_true):
    m_p = precision(y_pred, y_true)
    m_r = recall(y_pred, y_true)
    if m_p + m_r == 0:
        return 0.0
    return 2 * (m_p*m_r) / (m_p+m_r)

def macro_precision(y_pred, y_true):
    y_pred = y_pred.int()
    y_true = y_true.int()
    t_p = ((y_pred == 1) & (y_true == 1)).sum(dim=0)
    f_p = ((y_pred == 1) & (y_true == 0)).sum(dim=0)
    return t_p.float() / (t_p + f_p).float().clamp(min=1e-8)

def macro_recall(y_pred, y_true):
    y_pred = y_pred.int()
    y_true = y_true.int()
    t_p = ((y_pred == 1) & (y_true == 1)).sum(dim=0)
    f_n = ((y_pred == 0) & (y_true == 1)).sum(dim=0)
    return t_p.float() / (t_p + f_n).float().clamp(min=1e-8)

def macro_f1(y_pred, y_true):
    m_p = macro_precision(y_pred, y_true)
    m_r = recall(y_pred, y_true)
    return 2 * m_p * m_r / (m_p + m_r).clamp(min=1e-8)


In [17]:
device = 'cuda'

In [18]:
#x, y = next(iter(train_dataloader))
#y_p = net(x.to(device))
#y_d = y.to(device)
#recall(y_p, y_d), precision(y_p, y_d), jaccard_score(y_p, y_d), f1(y_p, y_d)

In [19]:
net = CNet().to(device)
train_model(net, train_dataloader, test_dataloader, 30, 1e-2)

Epoch 0 train loss: 0.5179343808237231
Epoch 1 train loss: 0.2725362771887127
Epoch 2 train loss: 0.25453745760354435
Epoch 3 train loss: 0.2418343865215037
Epoch 4 train loss: 0.23131749219199904
Epoch 5 train loss: 0.2239732694403528
Epoch 6 train loss: 0.2186850081487405
Epoch 7 train loss: 0.21434043829542707
Epoch 8 train loss: 0.2108110445642556
Epoch 9 train loss: 0.2076766268080547
Epoch 10 train loss: 0.20505094806106222
Epoch 11 train loss: 0.2022731054316194
Epoch 12 train loss: 0.19939743900807247
Epoch 13 train loss: 0.19664294971401272
Epoch 14 train loss: 0.19414420961857265
Epoch 15 train loss: 0.1920086242668794
Epoch 16 train loss: 0.18951937916223788
Epoch 17 train loss: 0.18757488022742008
Epoch 18 train loss: 0.18453345667458343
Epoch 19 train loss: 0.18244532956270298
Epoch 20 train loss: 0.18024083864784157
Epoch 21 train loss: 0.17825020841764513
Epoch 22 train loss: 0.17673604169175638
Epoch 23 train loss: 0.17463755593047897
Epoch 24 train loss: 0.173078863013

In [ ]:
a_j_score = 0.0
a_f1_score = 0.0
a_p_score = 0.0
a_r_score = 0.0

a_m_f1_score = None
a_m_p_score = None
a_m_r_score = None

for x_test, y_test in test_dataloader:
    val = (torch.sigmoid(net(x_test.to(device))) > 0.5).int()
    y_test = y_test.to(device).int()
    j_score = jaccard_score(val, y_test)
    f1_score = f1(val, y_test)
    p_score = precision(val, y_test)
    r_score = recall(val, y_test)
    m_f1_score = macro_f1(val, y_test)
    m_p_score = macro_precision(val, y_test)
    m_r_score = macro_recall(val, y_test)
    #print(j_score, f1_score, p_score, r_score)
    print(m_f1_score)
    #print(m_p_score)
    #print(m_r_score)
    if j_score > 0.4:
        print(val, y_test)

tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.5959, 0.0000, 0.5376, 0.7353, 0.3497,
        0.7353, 0.4237, 0.0000, 0.0000, 0.0000, 0.7353, 0.7353, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       device='cuda:0')
tensor([0.5988, 0.0000, 0.0000, 0.0000, 0.5208, 0.0000, 0.7042, 0.7042, 0.5208,
        0.7042, 0.7042, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       device='cuda:0')
tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,

In [21]:
def dig_accurate(y_preds, y_ground):
    
    y_preds[:, 4] = 0
    element_wise_eq = y_preds == y_ground

    for r in y_preds[torch.all(element_wise_eq, dim=1).int() == 1]:
        if (r.sum() > 0):
            print(r)
    print(y_preds[torch.all(element_wise_eq, dim=1).int() == 1][0])

    print(f"Row-wise Accuracy (perfect match): {sum(torch.all(element_wise_eq, dim=1).int())/element_wise_eq.shape[0]}")
    print(f"Element Wise Accuracy: {sum(element_wise_eq.int().flatten(0))/torch.numel(element_wise_eq)}")

In [22]:
dig_accurate((torch.sigmoid(net(X_test_torch.to(device))) > 0.5).int(), y_test_torch.to(device).int())

tensor([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       device='cuda:0', dtype=torch.int32)
tensor([0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       device='cuda:0', dtype=torch.int32)
tensor([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       device='cuda:0', dtype=torch.int32)
tensor([0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       device='cuda:0', dtype=torch.int32)
tensor([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       device='cuda:0', dtype=torch.int32)
tensor([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,